# 05 — Exploratory Data Analysis

EDA on the cleaned 4-institution Scopus dataset.

**Input:** `data/processed/merged_data.pkl`

## 1. Imports & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 80)
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')
%matplotlib inline

PROC_DIR = Path('../data/processed')
df = pd.read_pickle(PROC_DIR / 'merged_data.pkl')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

## 2. Citation Distribution (Target Variable)

In [ ]:
print('Citation count statistics:')
print(df['Citations'].describe().round(2))
print(f'\nSkewness : {df["Citations"].skew():.2f}')
print(f'Kurtosis : {df["Citations"].kurtosis():.2f}')
print(f'Zero-cited papers : {(df["Citations"] == 0).sum():,} ({(df["Citations"] == 0).mean()*100:.1f}%)')

In [ ]:
pcts = [10, 25, 50, 75, 90, 95, 99]
print('Citation percentiles:')
for p in pcts:
    v = df['Citations'].quantile(p/100)
    print(f'  {p:3d}th : {v:.0f}')

# Classification threshold candidates
print('\nThreshold candidates for high-impact classification:')
for p in [75, 80, 90]:
    t = df['Citations'].quantile(p/100)
    n = (df['Citations'] >= t).sum()
    print(f'  Top {100-p}%  →  >= {t:.0f} citations  ({n:,} papers)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Raw
axes[0].hist(df['Citations'].clip(upper=df['Citations'].quantile(0.99)),
             bins=80, edgecolor='none', alpha=0.8)
axes[0].set_title('Citations (raw, clipped at 99th pct)')
axes[0].set_xlabel('Citations')

# Log1p
axes[1].hist(np.log1p(df['Citations']), bins=60, edgecolor='none', alpha=0.8, color='steelblue')
axes[1].set_title('log1p(Citations)')
axes[1].set_xlabel('log1p(Citations)')

# ECDF
sorted_c = np.sort(df['Citations'])
axes[2].plot(sorted_c, np.arange(1, len(sorted_c)+1) / len(sorted_c))
axes[2].set_xscale('log')
axes[2].set_xlim(left=1)
axes[2].axhline(0.75, color='red', linestyle='--', label='75th pct')
axes[2].axhline(0.90, color='orange', linestyle='--', label='90th pct')
axes[2].legend()
axes[2].set_title('ECDF (log x-axis)')
axes[2].set_xlabel('Citations')

plt.tight_layout()
plt.savefig(PROC_DIR / 'eda_citation_dist.png', dpi=120)
plt.show()
print('Saved eda_citation_dist.png')

## 3. Publication Year

In [ ]:
year_counts = df['Year'].value_counts().sort_index()
print('Papers per year:')
print(year_counts.to_string())
print(f'\nTotal: {year_counts.sum():,}')

In [ ]:
year_stats = df.groupby('Year')['Citations'].agg(['count', 'median', 'mean']).round(1)
year_stats.columns = ['n_papers', 'median_cites', 'mean_cites']
print('Citations by year (note: recent years accumulate fewer citations):')
print(year_stats.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

year_stats['n_papers'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Papers per Year')
axes[0].set_xlabel('Year')
axes[0].tick_params(axis='x', rotation=45)

year_stats['median_cites'].plot(ax=axes[1], marker='o', color='tomato')
axes[1].set_title('Median Citations by Year')
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.savefig(PROC_DIR / 'eda_year.png', dpi=120)
plt.show()

## 4. Institution Breakdown

In [ ]:
inst_stats = df.groupby('institution')['Citations'].agg(['count', 'median', 'mean']).round(1)
inst_stats.columns = ['n_papers', 'median_cites', 'mean_cites']
inst_stats = inst_stats.sort_values('n_papers', ascending=False)
print(inst_stats.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Paper count per institution
inst_stats['n_papers'].sort_values().plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Papers per Institution')
axes[0].set_xlabel('Number of Papers')

# Citation distribution per institution (boxplot, log scale)
inst_order = inst_stats.sort_values('median_cites', ascending=False).index.tolist()
data_by_inst = [df.loc[df['institution'] == inst, 'Citations'].dropna() + 1 for inst in inst_order]
axes[1].boxplot(data_by_inst, labels=inst_order, vert=True, patch_artist=True,
                medianprops=dict(color='red', linewidth=2))
axes[1].set_yscale('log')
axes[1].set_title('Citation Distribution by Institution (log scale)')
axes[1].set_ylabel('Citations + 1')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(PROC_DIR / 'eda_institution.png', dpi=120)
plt.show()

## 5. Publication Type & Source Type

In [ ]:
print('Publication type distribution:')
pt = df['Publication type'].value_counts()
print(pt.to_string())

print('\nCitation stats by Publication type:')
print(df.groupby('Publication type')['Citations']
        .agg(['count', 'median', 'mean']).round(1)
        .sort_values('count', ascending=False).to_string())

In [ ]:
print('Source type distribution:')
print(df['Source type'].value_counts().to_string())

print('\nCitation stats by Source type:')
print(df.groupby('Source type')['Citations']
        .agg(['count', 'median', 'mean']).round(1)
        .sort_values('count', ascending=False).to_string())

In [ ]:
pt_stats = df.groupby('Publication type')['Citations'].agg(['count', 'median']).reset_index()
pt_stats.columns = ['pub_type', 'count', 'median_cites']
pt_stats = pt_stats.sort_values('count', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pt_stats.sort_values('count').plot(kind='barh', x='pub_type', y='count',
                                    ax=axes[0], legend=False, color='steelblue', edgecolor='none')
axes[0].set_title('Papers by Publication Type (top 10)')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('')

pt_stats.sort_values('median_cites').plot(kind='barh', x='pub_type', y='median_cites',
                                           ax=axes[1], legend=False, color='tomato', edgecolor='none')
axes[1].set_title('Median Citations by Publication Type')
axes[1].set_xlabel('Median Citations')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig(PROC_DIR / 'eda_pubtype.png', dpi=120)
plt.show()

## 6. Open Access

In [ ]:
# Fill missing with 'Unknown' for analysis
oa = df['Open access'].fillna('Unknown')
print('Open access categories:')
print(oa.value_counts().to_string())

print('\nCitation stats by Open access:')
temp = df.copy()
temp['OA'] = oa
print(temp.groupby('OA')['Citations']
        .agg(['count', 'median', 'mean']).round(1)
        .sort_values('median', ascending=False).to_string())

In [ ]:
oa_stats = temp.groupby('OA')['Citations'].agg(['count', 'median']).reset_index()
oa_stats.columns = ['oa_type', 'count', 'median_cites']
oa_stats = oa_stats.sort_values('median_cites', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

oa_stats.sort_values('count').plot(kind='barh', x='oa_type', y='count',
                                    ax=axes[0], legend=False, color='steelblue', edgecolor='none')
axes[0].set_title('Papers by Open Access Category')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('')

# Violin plot for top OA categories
top_oa = oa_stats.head(5)['oa_type'].tolist()
plot_data = [temp.loc[temp['OA'] == cat, 'Citations'].clip(upper=temp['Citations'].quantile(0.95))
             for cat in top_oa]
axes[1].violinplot(plot_data, positions=range(len(top_oa)), showmedians=True)
axes[1].set_xticks(range(len(top_oa)))
axes[1].set_xticklabels(top_oa, rotation=30, ha='right')
axes[1].set_title('Citation Distribution by Open Access (clipped at 95th pct)')
axes[1].set_ylabel('Citations')

plt.tight_layout()
plt.savefig(PROC_DIR / 'eda_open_access.png', dpi=120)
plt.show()

## 7. Venue Metrics

In [ ]:
venue_cols = [
    'CiteScore (publication year)',
    'SJR (publication year)',
    'SNIP (publication year)',
    'CiteScore percentile (publication year) *',
    'SJR percentile (publication year) *',
    'SNIP percentile (publication year) *',
    'Topic Prominence Percentile',
]
venue_cols = [c for c in venue_cols if c in df.columns]

print('Venue metric statistics:')
print(df[venue_cols].describe().round(2).to_string())

In [ ]:
# Correlation of venue metrics with log1p citations
temp = df[venue_cols].copy()
temp['log_citations'] = np.log1p(df['Citations'])

corr = temp.corr()['log_citations'].drop('log_citations').sort_values(ascending=False)
print('Pearson correlation with log1p(Citations):')
print(corr.round(3).to_string())

In [ ]:
base_metrics = [
    'CiteScore (publication year)',
    'SJR (publication year)',
    'SNIP (publication year)',
    'Topic Prominence Percentile',
]
base_metrics = [c for c in base_metrics if c in df.columns]

fig, axes = plt.subplots(1, len(base_metrics), figsize=(5*len(base_metrics), 4))
for ax, col in zip(axes, base_metrics):
    valid = df[[col, 'Citations']].dropna()
    ax.scatter(valid[col], np.log1p(valid['Citations']),
               alpha=0.05, s=5, rasterized=True)
    ax.set_xlabel(col.replace(' (publication year)', ''))
    ax.set_ylabel('log1p(Citations)')
    ax.set_title(col.replace(' (publication year)', ''))

plt.tight_layout()
plt.savefig(PROC_DIR / 'eda_venue_vs_citations.png', dpi=100)
plt.show()

## 8. Author Count

In [ ]:
print('Number of Authors statistics:')
print(df['Number of Authors'].describe().round(2))

print(f'\nSingle-author papers : {(df["Number of Authors"] == 1).sum():,}')
print(f'>10 authors          : {(df["Number of Authors"] > 10).sum():,}')
print(f'>50 authors (mega)   : {(df["Number of Authors"] > 50).sum():,}')

print('\nCorrelation (Pearson) of author count with log1p(Citations):')
v = df[['Number of Authors', 'Citations']].dropna()
print(np.corrcoef(v['Number of Authors'], np.log1p(v['Citations']))[0,1].round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Author count histogram (clipped)
auth = df['Number of Authors'].dropna()
axes[0].hist(auth.clip(upper=auth.quantile(0.99)), bins=50,
             edgecolor='none', alpha=0.8, color='steelblue')
axes[0].set_title('Author Count Distribution (clipped at 99th pct)')
axes[0].set_xlabel('Number of Authors')

# Median citations by author count bucket
df['author_bucket'] = pd.cut(df['Number of Authors'],
                              bins=[0, 1, 3, 6, 10, 20, 50, 10000],
                              labels=['1', '2-3', '4-6', '7-10', '11-20', '21-50', '50+'])
auth_cite = df.groupby('author_bucket', observed=True)['Citations'].median()
auth_cite.plot(kind='bar', ax=axes[1], color='tomato', edgecolor='none')
axes[1].set_title('Median Citations by Author Count')
axes[1].set_xlabel('Number of Authors')
axes[1].set_ylabel('Median Citations')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(PROC_DIR / 'eda_authors.png', dpi=120)
plt.show()

## 9. ASJC Field Classification

In [ ]:
asjc_col = 'All Science Journal Classification (ASJC) field name'

# Papers can have multiple ASJC fields (semicolon-separated) — explode them
asjc_series = df[asjc_col].dropna().str.split(';').explode().str.strip()
asjc_counts = asjc_series.value_counts()

print(f'Unique ASJC fields: {asjc_counts.nunique()}')
print('\nTop 20 ASJC fields by paper count:')
print(asjc_counts.head(20).to_string())

In [ ]:
# Median citations per top ASJC field (use first listed field per paper)
df['primary_asjc'] = df[asjc_col].str.split(';').str[0].str.strip()

asjc_stats = df.groupby('primary_asjc')['Citations'].agg(['count', 'median']).round(1)
asjc_stats.columns = ['n_papers', 'median_cites']
asjc_top = asjc_stats[asjc_stats['n_papers'] >= 100].sort_values('median_cites', ascending=False)

print('ASJC fields with ≥100 papers, sorted by median citations:')
print(asjc_top.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top fields by paper count (horizontal bar)
top15_count = asjc_stats.nlargest(15, 'n_papers').sort_values('n_papers')
top15_count['n_papers'].plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='none')
axes[0].set_title('Top 15 ASJC Fields by Paper Count')
axes[0].set_xlabel('Number of Papers')

# Top fields by median citations (shows field-level citation norm variation)
top15_cite = asjc_top.head(15).sort_values('median_cites')
top15_cite['median_cites'].plot(kind='barh', ax=axes[1], color='tomato', edgecolor='none')
axes[1].set_title('Top 15 ASJC Fields by Median Citations\n(fields with ≥100 papers)')
axes[1].set_xlabel('Median Citations')

plt.tight_layout()
plt.savefig(PROC_DIR / 'eda_asjc_count.png', dpi=120)
plt.show()

In [ ]:
# Boxplot: citation distribution across top fields — the key motivation for field normalization
top10_fields = asjc_top.head(10).index.tolist()
fig, ax = plt.subplots(figsize=(14, 5))

plot_data = [df.loc[df['primary_asjc'] == f, 'Citations'].dropna() + 1 for f in top10_fields]
bp = ax.boxplot(plot_data, labels=top10_fields, vert=True, patch_artist=True,
                medianprops=dict(color='red', linewidth=2),
                flierprops=dict(marker='.', alpha=0.2, markersize=3))
ax.set_yscale('log')
ax.set_title('Citation Distribution by ASJC Field (top 10 by median, log scale)\n'
             '→ Citation norms vary widely — justifies field-normalized target')
ax.set_ylabel('Citations + 1')
ax.tick_params(axis='x', rotation=40)

plt.tight_layout()
plt.savefig(PROC_DIR / 'eda_asjc_boxplot.png', dpi=120)
plt.show()

## 10. Abstract Length

In [ ]:
abs_len = df['Abstract'].dropna().str.split().str.len()
print('Abstract word count statistics:')
print(abs_len.describe().round(1))
print(f'\n<50 words  : {(abs_len < 50).sum():,}')
print(f'50-300 words: {((abs_len >= 50) & (abs_len <= 300)).sum():,}')
print(f'>300 words  : {(abs_len > 300).sum():,}')

# Correlation with citations
temp2 = df[['Abstract', 'Citations']].copy()
temp2['abs_len'] = temp2['Abstract'].str.split().str.len()
temp2 = temp2.dropna()
print(f'\nCorrelation abstract length vs log1p(Citations): '
      f'{np.corrcoef(temp2["abs_len"], np.log1p(temp2["Citations"]))[0,1]:.3f}')

## 11. Country / International Collaboration

In [ ]:
print('Number of Countries/Regions statistics:')
print(df['Number of Countries/Regions'].describe().round(2))

print(f'\nSingle-country papers  : {(df["Number of Countries/Regions"] == 1).sum():,}')
print(f'International (>1 ctry): {(df["Number of Countries/Regions"] > 1).sum():,}')

print('\nMedian citations by collaboration scope:')
df['collab_scope'] = pd.cut(df['Number of Countries/Regions'],
                             bins=[0, 1, 3, 100],
                             labels=['Domestic (1)', 'Regional (2-3)', 'Global (4+)'])
print(df.groupby('collab_scope', observed=True)['Citations']
        .agg(['count', 'median', 'mean']).round(1).to_string())

In [ ]:
collab_stats = df.groupby('collab_scope', observed=True)['Citations'].agg(['count', 'median']).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

collab_stats.plot(kind='bar', x='collab_scope', y='count', ax=axes[0],
                  legend=False, color='steelblue', edgecolor='none')
axes[0].set_title('Papers by Collaboration Scope')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)

collab_stats.plot(kind='bar', x='collab_scope', y='median', ax=axes[1],
                  legend=False, color='tomato', edgecolor='none')
axes[1].set_title('Median Citations by Collaboration Scope')
axes[1].set_xlabel('')
axes[1].set_ylabel('Median Citations')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(PROC_DIR / 'eda_collaboration.png', dpi=120)
plt.show()

## 12. Feature Correlation Heatmap

In [ ]:
num_cols = [
    'Citations',
    'Number of Authors',
    'Number of Countries/Regions',
    'CiteScore (publication year)',
    'SJR (publication year)',
    'SNIP (publication year)',
    'Topic Prominence Percentile',
    'CiteScore percentile (publication year) *',
]
num_cols = [c for c in num_cols if c in df.columns]

corr_mat = df[num_cols].corr()
short_names = {
    'CiteScore (publication year)': 'CiteScore',
    'SJR (publication year)': 'SJR',
    'SNIP (publication year)': 'SNIP',
    'Topic Prominence Percentile': 'Topic Prom.',
    'CiteScore percentile (publication year) *': 'CiteScore Pct',
    'Number of Authors': 'N Authors',
    'Number of Countries/Regions': 'N Countries',
}
corr_mat.columns = [short_names.get(c, c) for c in corr_mat.columns]
corr_mat.index   = [short_names.get(c, c) for c in corr_mat.index]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Pearson Correlation Matrix')
plt.tight_layout()
plt.savefig(PROC_DIR / 'eda_corr_heatmap.png', dpi=120)
plt.show()

print('\nCorrelations with Citations (sorted):')
print(corr_mat['Citations'].drop('Citations').sort_values(ascending=False).round(3).to_string())

## 13. Summary

In [ ]:
q75 = df['Citations'].quantile(0.75)
q90 = df['Citations'].quantile(0.90)

print('=' * 60)
print('EDA SUMMARY')
print('=' * 60)
print(f'Total papers          : {len(df):,}')
print(f'Year range            : {int(df["Year"].min())} – {int(df["Year"].max())}')
print(f'Institutions          : {df["institution"].nunique()}')
print(f'Citations  median/mean : {df["Citations"].median():.0f} / {df["Citations"].mean():.1f}')
print(f'Zero-cited            : {(df["Citations"]==0).mean()*100:.1f}%')
print(f'Top-25% threshold     : {q75:.0f} citations')
print(f'Top-10% threshold     : {q90:.0f} citations')
print(f'Abstract fill rate    : {df["Abstract"].notna().mean()*100:.1f}%')
print(f'Open access fill rate : {df["Open access"].notna().mean()*100:.1f}%')
print()
print('Key modelling notes:')
print('  • Citations are heavily right-skewed → use log1p transform for regression')
print('  • Top-25% threshold for binary classification is reasonable')
print('  • Venue metrics (CiteScore/SJR/SNIP) will be strong predictors')
print('  • Open access is 56% missing — treat as categorical with Unknown level')
print('  • Recent years (<2022) will have suppressed citation counts')